# 1. Dense Layers

### 1.1 A Single Neuron

Let us consider input to a single neuron. A neuron has a weight $w_i$ associated with each input $x_i$ and a bias $b$. Moreover, a neuron almost always has more than one input; the inputs of a single neuron are therefore contained in an row vector $X$ while the weights in $W$. The output of the given neuron $\hat{Y}$ is,

$$
\hat{Y} = f\!\left(b + \sum_{i=1}^N w_i X_i\right)
$$

In which, $N$ is the number of input connections. $f$ is our **activation function**. Activation functions are essential because they introduce non-linearity which allow neural networks to act as universal function approximators. They come in many forms, but one of the most simple let versatile activations functions is the rectified linear unit or **ReLU** which is often denoted as $\text{ReLU} (x)$. By defining the input vector $X = [X_1 X_2 ...]$ and the associted weights of the neuron $W = [W_1, W_2 ...]$, we can write the **forward pass** of a single neuron in terms of a dot product, 

$$ \hat{Y} = f \left(X \cdot W^T + b \right) $$

where $W^T$ is the transpose of $W$. Using NumPy we may implement this efficiently in Python,

In [2]:
import numpy as np

def ReLU(x):
    """This is the rectified linear unit (ReLU) which is one of the most common activiation functions
    used in machine learning."""
    return np.maximum(x, 0)

# For a single neuron,
layer_input = np.array([1, 2, 3])
weights = np.array([0.2, 0.8, -0.5])
bias = 2

# Computing the output,
neuron_output = ReLU(np.dot(layer_input, weights.T) + bias)
neuron_output

2.3

Note that the output of a single neuron (before passing through an activation function) is always a scalar value. However, in layers of a neural network, we have many neutrons and the ouput of a layer is an array. Therefore, the input of any activation function is performed element-wise on the ouput array of a layer. The **ReLU** activiation function is favoured for its effectiveness and versitility in neural network models despite its simplicity. It is defined by the following,

$$
\text{ReLU}(x) =
\begin{cases}
0 & \text{if } x < 0, \\
x & \text{if } x \geq 0.
\end{cases}
$$

### 1.2 Dense Layers

Now we start coding a **_Dense Layer_** of neurons where each neuron in the layer is connected to every neuron in the layer before it.  Let the output of the $j$-th neuron in the layer be denoted by $y_j$. It is given by:

$$
y_j = f \left( b_j + \sum_{i=1}^N X_i W_{ji} \right)
$$

Notice that the mathematical object $W_{ji}$, which holds the weights of the neurons, is now characterised by two indices. This means $W$ is a matrix (2D array in NumPy), with $j$ being the index for the $j$-th neuron and $i$ being the $i$-th weight associated with it. $W$ can be written as:

$$
W =
\begin{bmatrix}
    W_{11} & W_{12} & W_{13} & \dots  & W_{1N} \\
    W_{21} & W_{22} & W_{23} & \dots  & W_{2N} \\
    \vdots & \vdots & \vdots & \ddots & \vdots \\
    W_{M1} & W_{M2} & W_{M3} & \dots  & W_{MN}
\end{bmatrix}
$$

where $M$ is the number of neurons in the layer and $N$ is the number of inputs (or the number of neurons in the previous layer). We can imagine our weight matrix $W$ as the row vectors for the weights of each neuron stacked on top of each other. With this in mind, the shape of the weight matrix of a dense layer is $[M \times N]$. Also, since we are no longer talking about a single neuron, but a layer of them, the bias is not a single value but a row vector of length $M$ for each neuron:

$$
b =
\begin{bmatrix}
    b_{1} & b_{2} & b_{3} & \dots  & b_{M}
\end{bmatrix}
$$

The output of the layer $y$ is then a row vector of length $M$, given by the equation:

$$
y = f \left(XW^T + b \right)
$$

This is rather abstract, to make this more clear, let us consider the dot product $XW^T$,

$$
XW^T =

\begin{bmatrix}
    X_1 & X_2 & X_3 & \dots & X_N \\
\end{bmatrix}


\begin{bmatrix}
    W_{11} & W_{21} & W_{31} & \dots  & W_{M1} \\
    W_{12} & W_{22} & W_{23} & \dots  & W_{M2} \\
    \vdots & \vdots & \vdots & \ddots & \vdots \\
    W_{1N} & W_{2N} & W_{3N} & \dots  & W_{MN}
\end{bmatrix}

= [A_1, A_2, A_3, ..., A_M]
$$

The result of the row multiplication is a row vector with a length $M$ corresponding to the number of neurons in the layer. Of course, the output of our layer is then simply $\hat{y} = f(A + b)$. Since we have shown that a dense layer takes in an input row vector $X$ of length $N$ and transforms it to a vector $y$ of length $M$, we can simply imagine the layer as a function which applies a linear transformation that changes the dimensionality of its input. That is, $\text{DenseLayer}(X) = f(XW^T + b) = y$ where $X \in [N]$ and $y \in [M]$. Let us now implement this, 



**IMPORTANT**: The formalism we have used is consistent with PyTorch as seen in the documentation: https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html where the _dense_ layers are called _linear_ layers because they perfom the affine linear transformation $y = xA^T + b$.



In [5]:
# We have N=4 connections fed into the layer,
layer_input = np.array([1, 2, 3, 2.5])

# Weights matrix for three neurons,
weights = np.array([[0.2, 0.8, -0.5, 1.0],
                    [0.5, -0.91, 0.26, -0.5],
                    [-0.26, -0.27, 0.17, 0.87]])

# Bias row vector for the three neurons,
biases = np.array([2, 3, 0.5])

# Computing the output,
layer_output = ReLU(np.dot(layer_input, weights.T) + biases)
layer_output

array([4.8  , 1.21 , 2.385])

### 1.3 Generalising to Batches

It is very rare to give a neural network a single sample at a time in training. Instead, the NN is given multiple samples at a time. The number of samples given in a training cycle is the **batch size** B. We now replace the input row vector $X$ with a matrix of dimensions $[B \times N]$.



$$
XW^T =

\begin{bmatrix}
    X_{11} & X_{12} & X_{13} & \dots  & X_{1N} \\
    X_{21} & X_{22} & X_{23} & \dots  & X_{2N} \\
    \vdots & \vdots & \vdots & \ddots & \vdots \\
    X_{B1} & W_{B2} & X_{B3} & \dots  & X_{BN}
\end{bmatrix}


\begin{bmatrix}
    W_{11} & W_{21} & W_{31} & \dots  & W_{M1} \\
    W_{12} & W_{22} & W_{23} & \dots  & W_{M2} \\
    \vdots & \vdots & \vdots & \ddots & \vdots \\
    W_{1N} & W_{2N} & W_{3N} & \dots  & W_{MN}
\end{bmatrix}

$$

The result is the matrix $A$ which has the dimensions $[B \times M]$

In [11]:
input_array = np.array([[1, 2, 3, 2.5],
                        [2.5, 3, 2, 1],
                        [3, 2, 1, 2.5]])

weights = np.array([[0.2, 0.8, -0.5, 1.0],
                    [0.5, -0.91, 0.26, -0.5],
                    [-0.26, -0.27, 0.17, 0.87]])

biases = [2, 3, 0.5]

# Computing the output,
output_array = np.dot(input_array, weights.T)
output_array += biases
print(output_array)

[[4.8   1.21  2.385]
 [4.9   1.54  0.25 ]
 [6.2   1.69  1.525]]


### 1.4 Tensors